# Zero-Shot and Few-Shot Prompting

Large language models can perform many tasks from **instructions alone** — classify text, extract fields, translate languages, answer questions — without updating their weights. This is **zero-shot** prompting: zero labeled examples in the prompt. When you add one or more **input–output demonstrations** before the real question, the technique is called **few-shot** prompting (also **in-context learning**).

Few-shot prompting was central to GPT-3's impact: the same model could adapt to new tasks by reading examples in the context window. Today, frontier models are strong zero-shot performers on common tasks, but few-shot examples still improve accuracy, format compliance, and handling of niche labels or styles.

This notebook explains both techniques, how to design examples, trade-offs in token cost, and when each approach is appropriate. Live API demonstrations compare zero-shot and few-shot sentiment classification on the same reviews.

Prerequisites: **01. Introduction to Prompt Engineering**, **02. Anatomy of Effective Prompts**, and basic API usage from **03. LLM API Integration**.

---

## 1. Zero-Shot Prompting

**Zero-shot** means the model receives:

- A **task description** (instruction)
- Optionally **context** and **format** rules
- The **input** to process

...but **no completed examples** of input paired with the desired output.

### 1.1 How Zero-Shot Works

During pre-training and alignment, models see vast text that implicitly contains many task patterns — Q&A, summaries, classifications in forum posts, and more. At inference time, your instruction activates those patterns. You are not teaching a new skill from scratch; you are **steering** the model toward a task format it has seen in similar form.

Zero-shot quality depends on:

- **Clarity** of the instruction (see notebook 02)
- **Familiarity** of the task to the model
- **Model size and training** (larger models often zero-shot better)
- **Inference settings** (low temperature for classification)

### 1.2 When Zero-Shot Is Enough

| Situation | Why zero-shot works |
|-----------|---------------------|
| Common tasks (sentiment, summary, translation) | Well represented in training data |
| Simple label sets | Easy to describe in one line |
| Prototyping | Fast to iterate without curating examples |
| Strict token budget | No example overhead |

### 1.3 Zero-Shot Example (Text)

```
Classify the sentiment of the review as POSITIVE, NEGATIVE, or NEUTRAL.
Reply with only the label.

Review: The battery life is amazing but the screen is dim.
```

No prior review–label pairs appear in the prompt. The model infers the task from the instruction alone.

---

## 2. Few-Shot / In-Context Learning

**Few-shot** prompting includes **demonstrations** — typically one to several examples of correct behavior — before the actual input.

| Term | Number of examples |
|------|-------------------|
| Zero-shot | 0 |
| One-shot | 1 |
| Few-shot | Usually 2–10 (informal) |
| Many-shot | Dozens+ (long context models) |

### 2.1 Why Examples Help

Examples reduce ambiguity:

- **Label semantics** — what "NEUTRAL" means in *your* product reviews
- **Output format** — exact spelling, casing, JSON keys
- **Edge cases** — mixed positive/negative sentiment → NEUTRAL
- **Style** — terse labels vs explanations

The model does not permanently learn from these examples; they exist only in the **current context**. A new API call without examples behaves zero-shot again unless you resend them.

### 2.2 Two Ways to Encode Examples in Chat APIs

**Pattern A — Examples inside one user message** (common for batch classification):

```
Classify sentiment as POSITIVE, NEGATIVE, or NEUTRAL.

Examples:
Review: Loved it! -> POSITIVE
Review: Broke on day one. -> NEGATIVE

Review: {new review} ->
```

**Pattern B — Alternating user/assistant messages** (natural for chat):

```
user:      Classify: "Loved it!" ->
assistant: POSITIVE
user:      Classify: "Broke on day one." ->
assistant: NEGATIVE
user:      Classify: "{new review}" ->
```

Both patterns work; choose based on readability and provider conventions. Demonstrations below use **Pattern A** for compact comparison.

### 2.3 Number and Order of Examples

**How many?** There is no universal optimum. More examples can help up to a point, then:

- Diminishing accuracy gains
- Rising token cost
- Risk of **overfitting to example quirks** (e.g., all examples are short)

Start with **3–5 diverse examples**, measure on a held-out test set, then adjust.

**Order effects:** Models can be slightly sensitive to example order. For critical production prompts:

- Shuffle example order across tests
- Put the **hardest** or most representative examples last (closest to the real input)

**Format consistency:** Every example should use the **same template**. Mixed formats confuse the model.

```
# Good — consistent
Review: text A -> LABEL
Review: text B -> LABEL

# Weak — inconsistent
Review: text A -> LABEL
I felt sad about text B so NEGATIVE
```

---

## 3. Designing Effective Examples

Few-shot quality depends more on **example curation** than on example count.

### 3.1 Diversity

Cover the label space and common patterns:

- At least one example per class (for classification)
- Mixed sentiment phrasing (explicit praise, subtle complaint, mixed pros/cons)
- Length variation (one-word reviews and paragraphs)

Homogeneous examples teach a narrow pattern and fail on new phrasing.

### 3.2 Representativeness

Examples should resemble **production inputs**. If real reviews mention product SKUs and order numbers, include that in examples. Synthetic "toy" examples often fail on real data.

### 3.3 Label Leakage and Bias

**Label leakage** — accidentally signaling the answer in the instruction or example metadata:

- "Here is a clearly negative example: ..." (tells the model the label)
- Always putting NEGATIVE examples first (order bias)

**Bias in examples** — if all POSITIVE examples mention "food" and NEGATIVE mention "service," the model may learn the wrong shortcut. Review examples for spurious correlations.

### 3.4 Incorrect or Ambiguous Labels

Few-shot examples are **supervision**. Wrong labels teach wrong behavior. Ambiguous examples teach inconsistent behavior. Curate and review examples like training data — because for the duration of the request, they are.

---

## 4. When Few-Shot Helps or Hurts

| Few-shot helps | Few-shot hurts or is unnecessary |
|----------------|----------------------------------|
| Niche labels or custom taxonomies | Task is standard and zero-shot already works |
| Strict output format | Examples are noisy or mislabeled |
| Small models or weaker zero-shot | Token budget is tight |
| Reducing format parse errors | Too many long examples crowd out real input |

**Decision workflow:**

1. Try **zero-shot** with a clear instruction and format.
2. Measure on a **test set** of real inputs.
3. If errors cluster on format or edge cases, add **few-shot** examples targeting those failures.
4. Re-measure; remove examples that do not improve metrics.

For tasks requiring hundreds of examples worth of behavior change, consider **fine-tuning** instead of megabyte-sized prompts.

---

## 5. Demonstration: Zero-Shot Classification

Replace the placeholder API key before running. We classify product reviews with **no examples** in the prompt.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"

SYSTEM = (
    "You classify product review sentiment. "
    "Reply with exactly one word: POSITIVE, NEGATIVE, or NEUTRAL."
)

TEST_REVIEWS = [
    "Absolutely love this keyboard — typing feels incredible!",
    "Stopped working after two days. Waste of money.",
    "Battery is great but the display is too dim. Okay overall.",
    "It arrived on time. Nothing special to report.",
]


def zero_shot_classify(review: str) -> str:
    user = f"""Classify the sentiment of the review below.
Labels: POSITIVE, NEGATIVE, NEUTRAL
Reply with only the label.

Review: {review}"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": user},
        ],
        temperature=0.0,
        max_tokens=10,
    )
    return (response.choices[0].message.content or "").strip()


print("=== ZERO-SHOT RESULTS ===")
for review in TEST_REVIEWS:
    label = zero_shot_classify(review)
    print(f"{label:8} | {review[:60]}..." if len(review) > 60 else f"{label:8} | {review}")

Mixed reviews (pros and cons) are the hardest zero-shot case. Models may oscillate between POSITIVE and NEUTRAL. Few-shot examples clarifying **NEUTRAL** help.

---

## 6. Demonstration: Few-Shot Classification

The same reviews, now with **three demonstrations** before each classification request. Examples teach mixed-sentiment → NEUTRAL.

In [ ]:
FEW_SHOT_EXAMPLES = """
Review: Best purchase I made this year!
Label: POSITIVE

Review: Terrible quality, returned immediately.
Label: NEGATIVE

Review: Great camera but battery drains fast. Fine for the price.
Label: NEUTRAL
"""


def few_shot_classify(review: str) -> str:
    user = f"""Classify the sentiment of the review below.
Labels: POSITIVE, NEGATIVE, NEUTRAL
Use the same format as the examples.

Examples:
{FEW_SHOT_EXAMPLES}
Review: {review}
Label:"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": user},
        ],
        temperature=0.0,
        max_tokens=10,
    )
    return (response.choices[0].message.content or "").strip()


print("=== FEW-SHOT RESULTS ===")
for review in TEST_REVIEWS:
    label = few_shot_classify(review)
    print(f"{label:8} | {review[:60]}..." if len(review) > 60 else f"{label:8} | {review}")

---

## 7. Demonstration: Side-by-Side Comparison

Run zero-shot and few-shot on the same inputs and compare labels. Expected labels are provided for **learning** — in production you would measure against a labeled test set.

In [ ]:
EXPECTED = ["POSITIVE", "NEGATIVE", "NEUTRAL", "NEUTRAL"]

print(f"{'Review (short)':<42} {'Expected':<10} {'Zero':<10} {'Few':<10}")
print("-" * 74)

zero_correct = few_correct = 0

for review, expected in zip(TEST_REVIEWS, EXPECTED):
    z = zero_shot_classify(review)
    f = few_shot_classify(review)
    zero_correct += int(z.upper().startswith(expected))
    few_correct += int(f.upper().startswith(expected))
    short = (review[:39] + "...") if len(review) > 42 else review
    print(f"{short:<42} {expected:<10} {z:<10} {f:<10}")

n = len(TEST_REVIEWS)
print("\nAccuracy (this tiny set):")
print(f"  Zero-shot: {zero_correct}/{n}")
print(f"  Few-shot:  {few_correct}/{n}")

On a four-review demo, results may tie or differ by one label. The point is the **method**: few-shot targets errors you saw in zero-shot by showing boundary cases. Always evaluate on dozens or hundreds of real examples before shipping.

---

## 8. Demonstration: Few-Shot via Message History (Pattern B)

The same task using **alternating user/assistant** messages — closer to how chat UIs work.

In [ ]:
demo_pairs = [
    ("Best purchase I made this year!", "POSITIVE"),
    ("Terrible quality, returned immediately.", "NEGATIVE"),
    ("Great camera but battery drains fast.", "NEUTRAL"),
]

new_review = "Battery is great but the display is too dim. Okay overall."

messages = [
    {
        "role": "system",
        "content": "Classify review sentiment. Reply with one word: POSITIVE, NEGATIVE, or NEUTRAL.",
    }
]

for text, label in demo_pairs:
    messages.append({"role": "user", "content": f"Review: {text}"})
    messages.append({"role": "assistant", "content": label})

messages.append({"role": "user", "content": f"Review: {new_review}"})

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.0,
    max_tokens=10,
)

print("Review:", new_review)
print("Label:", response.choices[0].message.content)

Pattern B costs more tokens (full messages per example) but mirrors conversational fine-tuning data. Use it when the application already maintains chat history.

---

## 9. Token Cost of Few-Shot

Every example adds **input tokens** on every request. Rough planning:

$$\text{prompt tokens} \approx \text{instruction} + \sum \text{example tokens} + \text{input tokens}$$

If you send 5 examples × 50 tokens each, you pay for 250 extra tokens **per classification** — multiplied by millions of requests in production.

**Mitigations:**

- Use the **minimum** number of examples that reach target accuracy
- Keep examples **short** but representative
- Cache static example prefixes where providers support prompt caching
- Fall back to zero-shot for easy cases (router pattern — Advanced Topics)

Few-shot is not free accuracy; it is a trade between token spend and error rate.

---

## 10. Summary

**Zero-shot** prompting relies on instructions alone — fast, cheap, and often sufficient for common tasks on strong models.

**Few-shot** prompting adds demonstrations in the context window (**in-context learning**), teaching label meaning, format, and edge cases without weight updates. Examples must be **diverse**, **consistent**, and **correctly labeled**.

Encode examples in a **single user block** or as **user/assistant turns**; keep formats uniform. Start zero-shot, measure errors, add targeted examples, re-measure.

The demonstrations classified the same reviews under zero-shot and few-shot and compared accuracy on a tiny labeled set. Next: **04. Chain-of-Thought and Reasoning Prompts** — eliciting step-by-step reasoning for harder problems.